---
# **LAB 3 - CUDA Execution Model**
---

# ▶️ CUDA tools...

In [ ]:
!nvidia-smi

In [ ]:
import numpy as np
import numba
from numba import cuda
import warnings
warnings.filterwarnings("ignore")

print(np.__version__)
print(numba.__version__)

cuda.detect()



In [ ]:
# Suppress Numba deprecation and performance warnings
from numba.core.errors import NumbaDeprecationWarning, NumbaPerformanceWarning
import warnings

warnings.simplefilter('ignore', category=NumbaDeprecationWarning)
warnings.simplefilter('ignore', category=NumbaPerformanceWarning)

Utils for compiling and running Numba CUDA code.

In [ ]:
from numba import cuda

def mem_snapshot():
    free, total = cuda.current_context().get_memory_info()
    return total - free, free, total

# Quick device spec report (Numba)
def print_device_info():
    dev = cuda.get_current_device()   # raises if no CUDA device
    ctx = cuda.current_context()

    print("Device object repr:", dev)
    print("Device name:          ", getattr(dev, "name", "<unknown>"))
    print("Compute capability:   ", getattr(dev, "compute_capability", "<unknown>"))

    # Common numeric properties (use getattr to avoid attribute errors)
    props = {
        "  multi_processor_count": ["MULTIPROCESSOR_COUNT"],
        "  max_threads_per_block": ["MAX_THREADS_PER_BLOCK"],
        "  max_block_dim_x":       ["MAX_BLOCK_DIM_X"],
        "  total_memory (bytes)":  ["total_memory"],
        "  shared_memory_per_block (bytes)": ["MAX_SHARED_MEMORY_PER_BLOCK"],
        "  warp_size":             ["WARP_SIZE"],
    }

    for label, keys in props.items():
        val = None
        for k in keys:
            val = getattr(dev, k, None)
            if val is not None:
                break
        print(f"{label:33}: {val}")

print_device_info()

# ✅ Parallel Reduction

In [ ]:
import numpy as np
from numba import cuda
import time

from numba import cuda
        
@cuda.jit
def blockParReduce(array, out):
    tid = cuda.threadIdx.x
    idx = cuda.grid(1) # global index
    n = len(array)

    # boundary check (matches CUDA C)
    if idx >= n:
        return

    # pointer to this block's segment (logical equivalent to: in + blockIdx.x * blockDim.x)
    block_skip = cuda.blockIdx.x * cuda.blockDim.x

    # in-place reduction in global memory (interleaved / no divergence schema)
    # in-place reduction in global memory (stride doubles each step)
    stride = 1
    while stride < cuda.blockDim.x:
        if (tid % (2 * stride)) == 0:
            array[block_skip + tid] += array[block_skip + tid + stride]
        cuda.syncthreads()
        stride *= 2

    # write one value per block
    if tid == 0:
        out[cuda.blockIdx.x] = array[block_skip]


# ----------------------------
# host-side usage
# ----------------------------
blockSize = 1024;               # block dim 1D
numBlock = 1024*1024          # grid dim 1D
n = blockSize * numBlock;       # array dim

# prepare data
a = np.ones(n, dtype=np.int32)
a_d = cuda.to_device(a)
b_d = cuda.device_array(numBlock, dtype=np.int32)

# numpy sum time
tic = time.time()
s_cpu = a.sum()
toc = time.time()
print(f"Numpy sum time: {toc - tic:.4f} seconds")

# launch kernel
t0 = time.perf_counter()
blockParReduce[numBlock, blockSize](a_d, b_d)
cuda.synchronize()
t1 = time.perf_counter()
print(f"Kernel execution time: {t1 - t0:.4f} seconds")
print("speedup over numpy:", (toc - tic) / (t1 - t0))

# copy result back to host
b = b_d.copy_to_host()
s_gpu = b.sum()
assert s_cpu == s_gpu, "Error! Reduction result does not match!"

# print GPU memory info
used, free, total = mem_snapshot()
print("\nMemory occupancy:")
print(f"    GPU total: {total/1e9:.3f} GB")
print(f"    GPU free : {free/1e9:.3f} GB")
print(f"    GPU used : {used/1e9:.3f} GB")

## ↘️ TODO...

**Background: Divergence in Reduction**

-   Problem:
    -   threads in the same warp take different paths
    -   warps execute both paths (masked execution)
    -   performance drops
-   Goal:
    -   restructure indexing so the condition is based on a contiguous range of thread IDs

- Divergence-Avoiding Idea

    -  Instead of checking tid % (2\*stride) == 0, compute a new local index:
    $$
    index = 2 \cdot stride \cdot tid
    $$

    -   Then only threads with:
    $$
    index < blockDim.x
    $$


-   Implement the No-Divergence Reduction Loop

-   Use:

    -   `stride = 1, 2, 4, ...`
    -   `index = 2 * stride * tid`
    -   update: `in_arr[base + index] += in_arr[base + index + stride]`

Template:

```{python}
@cuda.jit
def blockParReduce_no_div(in_arr, out_arr, n):
    pass
```

## ➡️ Solution...

In [ ]:
import numpy as np
from numba import cuda
import time

from numba import cuda
        
@cuda.jit
def blockParReduce_no_div(array, out):
    tid = cuda.threadIdx.x
    idx = cuda.grid(1) # global index
    n = len(array)

    # boundary check (matches CUDA C)
    if idx >= n:
        return

    # pointer to this block's segment (logical equivalent to: in + blockIdx.x * blockDim.x)
    block_skip = cuda.blockIdx.x * cuda.blockDim.x

    # in-place reduction in global memory (interleaved / no divergence schema)
    stride = 1
    while stride < cuda.blockDim.x:
        index = 2 * stride * tid
        if index < cuda.blockDim.x:
            array[block_skip + index] += array[block_skip + index + stride]
        cuda.syncthreads() # synchronize within block
        stride *= 2

    # write one value per block
    if tid == 0:
        out[cuda.blockIdx.x] = array[block_skip]


# ----------------------------
# host-side usage
# ----------------------------
blockSize = 1024;               # block dim 1D
numBlock = 1024*1024          # grid dim 1D
n = blockSize * numBlock;       # array dim

# prepare data
a = np.ones(n, dtype=np.int32)
a_d = cuda.to_device(a)
b_d = cuda.device_array(numBlock, dtype=np.int32)

# numpy sum time
tic = time.time()
s_cpu = a.sum()
toc = time.time()
print(f"Numpy sum time: {toc - tic:.4f} seconds")

# launch kernel
t0 = time.perf_counter()
blockParReduce_no_div[numBlock, blockSize](a_d, b_d)
cuda.synchronize()
t1 = time.perf_counter()
print(f"Kernel execution time: {t1 - t0:.4f} seconds")
print("speedup over numpy:", (toc - tic) / (t1 - t0))

# copy result back to host
b = b_d.copy_to_host()
s_gpu = b.sum()
assert s_cpu == s_gpu, "Error! Reduction result does not match!"

# print GPU memory info
used, free, total = mem_snapshot()
print("\nMemory occupancy:")
print(f"    GPU total: {total/1024**3:.3f} GB")
print(f"    GPU free : {free/1024**3:.3f} GB")
print(f"    GPU used : {used/1024**3:.3f} GB")

# ✅ Image histogram

In [ ]:
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
        
img = Image.open("../images/dog.png") # load image
img_mat = np.array(img).astype(np.uint8) # convert to numpy array
H, W, C = img_mat.shape
print(f"Image size: {H} x {W} x {C}")
img

## ↘️ TODO...

**Problem Description**

-   Given an RGB image `image` of shape: $(H, W, 3)$

-   compute a histogram such that:

    -   `histogram[0, i]` = num pixels with **red value** `i`
    -   `histogram[1, i]` = num pixels with **green value** `i`
    -   `histogram[2, i]` = num pixels with **blue value** `i`

-   Each color channel has **256 bins**

🔹 **CPU Reference Function**

-   CPU helper function that computes a frequency histogram for a 1D array:

``` python
def array_freq(arr):
    h = np.zeros(256, dtype=np.int32)
    for e in arr:
        h[e] += 1
    return h
```

🔹 **CUDA Kernel Requirements**

1.  Uses a 2D grid and 2D block
2.  Maps each thread to one pixel $(y, x)$
3.  Reads the pixel’s RGB values
4.  Updates the histogram using atomic additions
5.  Avoids out-of-bounds accesses

``` python
from numba import cuda

@cuda.jit
def histGPU(image, histogram):
    """
    image: uint8 array of shape (H, W, 3)
    histogram: int32 array of shape (3, 256)
    """
    # TODO
    
```

## ➡️ Solution...

In [ ]:
import numpy as np
from numba import cuda

def array_freq(arr):
    h = np.zeros(256, dtype=np.int32)
    for e in arr:
        h[e] += 1
    return h


@cuda.jit
def histGPU(image, histogram):
    """
    Compute color histogram of an image using GPU.
    
    Parameters:
        image: uint8 array of shape (height, width, 3)
        histogram: int32 array of shape (3, 256)
    """
    
    y, x = cuda.grid(2)

    H, W, C = image.shape
    if x >= W or y >= H:
        return

    R = image[y, x, 0]
    G = image[y, x, 1]
    B = image[y, x, 2]

    cuda.atomic.add(histogram, (0, R), 1)
    cuda.atomic.add(histogram, (1, G), 1)
    cuda.atomic.add(histogram, (2, B), 1)


# ----------------------------
# Run kernel histGPU 
# ----------------------------

# histogram and image allocation on device
hist = np.zeros((3, 256), dtype=np.int32)
d_img = cuda.to_device(img_mat)
d_hist = cuda.device_array_like(hist)

# parameters for kernel launch
threads = (16, 16)
blocks = (
    (H + threads[0] - 1) // threads[0],
    (W + threads[1] - 1) // threads[1],
)

# launch kernel
histGPU[blocks, threads](d_img, d_hist)
cuda.synchronize()

# copy result back to host
hist = d_hist.copy_to_host()

# check results
hR = array_freq(img_mat[:, :, 0].flatten())
hG = array_freq(img_mat[:, :, 1].flatten())
hB = array_freq(img_mat[:, :, 2].flatten())
print("R bins diff:", (hist[0]-hR).sum())
print("G bins diff:", (hist[1]-hG).sum())
print("B bins diff:", (hist[2]-hB).sum())

print("\nHistogram bins (R, G, B):")
for i in range(256):
    print(f"{i:3d}: {hist[0, i]:6d} {hist[1, i]:6d} {hist[2, i]:6d}")
    